In [ ]:
import dapla as dp
from fagfunksjoner import ProjectRoot
import pandas as pd
with ProjectRoot():
    import src.timeseries as ts

In [ ]:
offbal_df = dp.read_pandas(
    "gs://ssb-prod-finregn-data-produkt/inndata/offentlig/offbal_test.parquet",
    file_format="parquet",
)
offbal_df

In [ ]:
serie_cols = [
    "sek_esa",
    "acc_entry",
    "objekt",
    "flow_stock",
    "func_cat",
    "maturity",
    "land",
]

In [ ]:
offbal_df["serienavn"] = offbal_df[serie_cols].apply(lambda row: ".".join(row), axis=1)
offbal_df

In [ ]:
offbal_tid = (
    offbal_df[["new_datetime", "serienavn", "verdi"]]
    .pivot(index="new_datetime", columns="serienavn")
    .fillna(0)
    .droplevel(0, axis=1)
)

# offbal_tid.index = pd.PeriodIndex(offbal_tid.index, freq="Q")
offbal_tid.columns.name = "periode"

offbal_tid.index = pd.PeriodIndex(offbal_tid.index, freq="Q")
offbal_tid.index.name = None
offbal_tid#.reset_index().set_index('new_datetime', drop=True)  # .transpose()

In [ ]:
offbal_tid[(offbal_tid.index >= "2022Q1") & (offbal_tid.index <= "2022Q4")]

In [ ]:
yearlist = sorted(list(set([p.split('Q')[0] for p in offbal_tid.index.astype(str).tolist()])))
yearlist

In [ ]:
yearlist[:-3]

In [ ]:
datalist = []
for year in sorted(list(set([p.split('Q')[0] for p in offbal_tid.index.astype(str).tolist()]))):
    d = offbal_tid[(offbal_tid.index >= f"{year}Q1") & (offbal_tid.index <= f"{year}Q4")]
    datalist.append(d)

In [ ]:
startdata = pd.concat(datalist[:-3])
startdata

In [ ]:
datadict = {}
for year in sorted(list(set([p.split('Q')[0] for p in offbal_tid.index.astype(str).tolist()]))):
    datadict[year] = offbal_tid[(offbal_tid.index >= f"{year}Q1") & (offbal_tid.index <= f"{year}Q4")]

In [ ]:
datadict.keys()

In [ ]:
data21 = datadict['2021']
data22 = datadict['2022']
data23 = datadict['2023']

In [ ]:
offbal_tid.T